# 🧠 SentimentIQ — AI Sentiment Analysis Dashboard
> **Run this notebook in Google Colab.** Each cell is self-contained and annotated.

---
## Overview
This notebook implements a complete **NLP Sentiment Analysis Pipeline** with:
- Text preprocessing (NLTK, spaCy)
- Multi-engine sentiment analysis (VADER + TextBlob ensemble)
- Beautiful interactive visualizations (Plotly + WordCloud)
- A full Streamlit dashboard launcher (via `localtunnel`)


## ① Install Dependencies

In [ ]:
# Install all required packages
!pip install -q streamlit nltk textblob vaderSentiment plotly wordcloud pyngrok
!pip install -q scikit-learn pandas numpy matplotlib
print('✅ All packages installed!')

## ② Download NLTK Data & NLP Corpora

In [ ]:
import nltk
for corpus in ['vader_lexicon', 'stopwords', 'punkt', 'wordnet',
               'punkt_tab', 'averaged_perceptron_tagger']:
    nltk.download(corpus, quiet=True)
print('✅ NLTK data downloaded!')

## ③ Core NLP Functions (Preprocessing + Sentiment Engine)

In [ ]:
"""
╔══════════════════════════════════════════════════╗
║  SENTIMENT ANALYSIS ENGINE — Core Implementation ║
╚══════════════════════════════════════════════════╝
"""

import re
import pandas as pd
import numpy as np
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

# Initialize NLP tools
vader      = SentimentIntensityAnalyzer()
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))


# ── 1. Text Preprocessing ──────────────────────────────────────
def preprocess_text(text: str) -> str:
    """Full NLP preprocessing pipeline.
    
    Steps:
    1. Lowercase conversion
    2. Remove URLs, mentions, hashtags
    3. Remove punctuation & digits
    4. Tokenization
    5. Stopword removal
    6. Lemmatization
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove noise
    text = re.sub(r'http\S+|www\S+', '', text)    # URLs
    text = re.sub(r'@\w+|#\w+', '', text)         # @mentions, #hashtags
    
    # Step 3: Remove punctuation/digits
    text = re.sub(r"[^a-z\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 4 + 5 + 6: Tokenize → filter stopwords → lemmatize
    try:
        tokens = nltk.word_tokenize(text)
    except Exception:
        tokens = text.split()
    
    tokens = [
        lemmatizer.lemmatize(tok)
        for tok in tokens
        if tok not in STOP_WORDS and len(tok) > 2
    ]
    return ' '.join(tokens)


# ── 2. Sentiment Classifier ────────────────────────────────────
def analyze_sentiment(text: str, engine: str = 'VADER+TextBlob') -> dict:
    """Classify text sentiment using selected engine.
    
    Engines:
    - 'VADER'         : Best for social media text
    - 'TextBlob'      : Best for formal/editorial text  
    - 'VADER+TextBlob': Weighted ensemble (default)
    
    Returns dict with: label, polarity, confidence, scores
    """
    clean = preprocess_text(text)
    if not clean.strip():
        clean = text  # Fallback to raw text
    
    # --- VADER Scores ---
    vs            = vader.polarity_scores(text)  # Use original for VADER
    vader_compound = vs['compound']
    
    # --- TextBlob Scores ---
    tb             = TextBlob(text)
    tb_polarity    = tb.sentiment.polarity       # Range: -1.0 to +1.0
    tb_subjectivity= tb.sentiment.subjectivity  # Range: 0 (obj) to 1 (subj)
    
    # --- Compute final polarity ---
    if engine == 'VADER+TextBlob':
        compound = (vader_compound * 0.6) + (tb_polarity * 0.4)
    elif engine == 'VADER':
        compound = vader_compound
    else:  # TextBlob only
        compound = tb_polarity
    
    # --- Classify ---
    if compound >= 0.05:
        label, emoji = 'POSITIVE', '✅'
    elif compound <= -0.05:
        label, emoji = 'NEGATIVE', '❌'
    else:
        label, emoji = 'NEUTRAL', '➖'
    
    # Confidence: scaled abs polarity
    confidence = min(99, round(abs(compound) * 100 + 50))
    
    return {
        'text'          : text,
        'clean_text'    : clean,
        'label'         : label,
        'emoji'         : emoji,
        'polarity'      : round(compound, 4),
        'confidence'    : confidence,
        'subjectivity'  : round(tb_subjectivity, 4),
        'vader_pos'     : round(vs['pos'], 3),
        'vader_neg'     : round(vs['neg'], 3),
        'vader_neu'     : round(vs['neu'], 3),
        'vader_compound': round(vader_compound, 4),
        'tb_polarity'   : round(tb_polarity, 4),
    }


# ── 3. Batch Analyzer ─────────────────────────────────────────
def batch_analyze(texts: list, engine: str = 'VADER+TextBlob') -> pd.DataFrame:
    """Analyze a list of texts; returns tidy DataFrame."""
    rows = []
    for i, text in enumerate(texts, 1):
        if not isinstance(text, str) or not text.strip():
            continue
        r = analyze_sentiment(text, engine)
        rows.append({
            'Index'         : i,
            'Text'          : text[:100] + ('…' if len(text) > 100 else ''),
            'Sentiment'     : r['label'],
            'Polarity'      : r['polarity'],
            'Confidence %'  : r['confidence'],
            'Subjectivity'  : r['subjectivity'],
            'VADER Compound': r['vader_compound'],
        })
    return pd.DataFrame(rows)

print('✅ Core functions loaded!')

## ④ Test Single Text Analysis

In [ ]:
# ── Test with sample sentences ────────────────────────────────
test_texts = [
    'This product is absolutely amazing! Best purchase I have made all year.',
    'Terrible quality. Broke after two days. Complete waste of money.',
    'The package arrived today. It contains the items I ordered.',
    'Great food but the service was incredibly slow and rude.',
    'I love this! 😍 Highly recommend to everyone!',
]

print('=' * 60)
print('  SENTIMENT ANALYSIS RESULTS')
print('=' * 60)

for text in test_texts:
    result = analyze_sentiment(text)
    print(f"\nText     : {text[:65]}..." if len(text) > 65 else f"\nText     : {text}")
    print(f"Sentiment: {result['emoji']} {result['label']}")
    print(f"Polarity : {result['polarity']:+.4f}  |  Confidence: {result['confidence']}%  |  Subjectivity: {result['subjectivity']:.4f}")
    print(f"VADER +/-/= : {result['vader_pos']} / {result['vader_neg']} / {result['vader_neu']}")

## ⑤ Batch Analysis on Sample Dataset

In [ ]:
# ── Sample datasets ───────────────────────────────────────────
PRODUCT_REVIEWS = [
    'This product is absolutely amazing! Best purchase I made all year.',
    'Terrible quality. Broke after two days. Complete waste of money.',
    "It's okay, nothing special. Does the job but could be better.",
    'Exceeded all my expectations! Fast delivery and perfect packaging.',
    'Disappointed with this item. The description was very misleading.',
    'Works fine. Average product for the price range.',
    'Phenomenal! Highly recommend to everyone looking for quality.',
    'Worst purchase ever. Customer service was also unhelpful.',
    'Decent product but nothing extraordinary. Would consider again.',
    'Outstanding quality! Will definitely buy from this seller again.',
    'Not worth the price at all. Very cheap build quality.',
    'Exactly as described. Happy with my purchase overall.',
]

TWITTER_COMMENTS = [
    'Just watched the new movie - absolutely breathtaking! #MustWatch',
    "Can't believe how bad the service was today. Never going back!",
    'Today was just another Monday. Nothing exciting happened.',
    'The concert last night was INCREDIBLE! Best night of my life!',
    'Traffic is terrible as usual. Why do I even bother taking this route?',
    "Meh, the new update didn't really change anything important.",
    'So grateful for all the support from this amazing community!',
    'This app keeps crashing. So frustrating and such poor quality.',
    'Had lunch. It was fine. Going back to work now.',
    'Just got promoted! Dreams really do come true if you work hard!',
]

# Run batch analysis
all_texts = PRODUCT_REVIEWS + TWITTER_COMMENTS
df = batch_analyze(all_texts)

print('\nBatch Analysis Results:')
print(df.to_string(index=False))

print(f"\n{'='*50}")
print('SUMMARY')
print(f"{'='*50}")
vc = df['Sentiment'].value_counts()
for sent, count in vc.items():
    pct = count / len(df) * 100
    print(f"{sent:10s}: {count:3d}  ({pct:.0f}%)")
print(f"\nAverage Polarity  : {df['Polarity'].mean():+.4f}")
print(f"Average Confidence: {df['Confidence %'].mean():.1f}%")

## ⑥ Visualizations with Plotly

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

PALETTE   = {'POSITIVE': '#10d48e', 'NEGATIVE': '#ff4d6d', 'NEUTRAL': '#6b9fff'}
DARK_BG   = '#111827'
CARD_BG   = '#1a2235'
TEXT_C    = '#e8edf8'
MUTED_C   = '#8899bb'
GRID_C    = '#2a3550'

BASE = dict(
    paper_bgcolor=DARK_BG, plot_bgcolor=DARK_BG,
    font=dict(family='Arial', color=TEXT_C),
    margin=dict(l=40, r=40, t=60, b=40),
)

# ── Subplot dashboard ─────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sentiment Distribution', 'Sentiment Count',
                    'Polarity Histogram', 'Polarity vs Confidence'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'histogram'}, {'type': 'scatter'}]]
)

# 1. Pie chart
vc_counts = df['Sentiment'].value_counts()
fig.add_trace(go.Pie(
    labels=vc_counts.index.tolist(),
    values=vc_counts.values.tolist(),
    marker_colors=[PALETTE[s] for s in vc_counts.index],
    hole=0.5,
), row=1, col=1)

# 2. Bar chart
for label in ['POSITIVE', 'NEGATIVE', 'NEUTRAL']:
    cnt = vc_counts.get(label, 0)
    fig.add_trace(go.Bar(
        x=[label], y=[cnt],
        name=label, marker_color=PALETTE[label],
        showlegend=False,
    ), row=1, col=2)

# 3. Polarity histogram
for label, color in PALETTE.items():
    sub = df[df['Sentiment'] == label]['Polarity']
    if len(sub):
        fig.add_trace(go.Histogram(
            x=sub, name=label, marker_color=color,
            opacity=0.75, showlegend=False,
        ), row=2, col=1)

# 4. Scatter plot
for label, color in PALETTE.items():
    sub = df[df['Sentiment'] == label]
    if len(sub):
        fig.add_trace(go.Scatter(
            x=sub['Polarity'], y=sub['Confidence %'],
            mode='markers', name=label,
            marker=dict(color=color, size=10, opacity=0.8),
            showlegend=False,
        ), row=2, col=2)

fig.update_layout(
    **BASE,
    title=dict(text='🧠 SentimentIQ — Analysis Dashboard', font_size=18),
    height=700, barmode='overlay',
)
fig.update_annotations(font_color=MUTED_C)
fig.show()

## ⑦ Word Cloud

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import random

# ── Generate separate word clouds for each sentiment ──────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0a0e1a')

color_maps = {
    'POSITIVE': ['#10d48e','#0aad74','#05ffb0','#b0ffe0'],
    'NEGATIVE': ['#ff4d6d','#ff1a40','#ff8098','#ffb3bf'],
    'NEUTRAL':  ['#6b9fff','#3d7eff','#a0bfff','#c8daff'],
}

for ax, (sentiment, colors) in zip(axes, color_maps.items()):
    subset = df[df['Sentiment'] == sentiment]
    
    # Join and preprocess all texts
    combined = ' '.join([
        preprocess_text(t) for t in all_texts
        if analyze_sentiment(t)['label'] == sentiment
    ])
    
    if not combined.strip():
        combined = sentiment.lower() + ' text'
    
    def make_color_func(c_list):
        def color_func(*args, **kwargs):
            return random.choice(c_list)
        return color_func
    
    wc = WordCloud(
        width=500, height=300,
        background_color='#1a2235',
        max_words=60,
        color_func=make_color_func(colors),
        collocations=False,
    ).generate(combined)
    
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_facecolor('#1a2235')
    ax.set_title(f'{sentiment} ({len(subset)} texts)',
                 color=colors[0], fontsize=13, fontweight='bold', pad=10)

plt.suptitle('Word Cloud by Sentiment', color='#e8edf8',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('wordcloud_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0e1a')
plt.show()
print('✅ Word cloud saved!')

## ⑧ Launch the Full Streamlit Dashboard
> This writes the `app.py` file and launches via `localtunnel`. Click the URL that appears below.

In [ ]:
# Write the full app.py (from the provided file or inline)
# If you uploaded app.py to Colab, this step is already done.
# Otherwise, the app content is written below.

import os
if not os.path.exists('app.py'):
    print('⚠️ app.py not found — please upload it to Colab first!')
    print('  Files → Upload → select app.py')
else:
    print('✅ app.py found!')

# Install localtunnel for Colab public URL
!npm install -q localtunnel
print('✅ localtunnel installed!')

In [ ]:
import subprocess, threading, time

# Launch Streamlit in background
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(4)  # Wait for server to start

# Get public URL via localtunnel
tunnel = subprocess.Popen(
    ['npx', 'localtunnel', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(3)
try:
    line = tunnel.stdout.readline().decode().strip()
    print(f'\n🚀 Dashboard is live at: {line}')
    print('   Click the URL above to open SentimentIQ!')
    print('   (You may need to click "Click to Continue" on the tunnel page)')
except Exception:
    print('Streamlit running at: http://localhost:8501')
    print('Use port forwarding or localtunnel manually to access externally.')

---
## ⑨ Standalone Usage (No Streamlit)
Use the functions directly from this notebook:

In [ ]:
# ── Quick standalone sentiment check ─────────────────────────
user_text = input('Enter text to analyze: ')

if user_text.strip():
    result = analyze_sentiment(user_text)
    print(f'\n{"="*50}')
    print(f'  Sentiment  : {result["emoji"]} {result["label"]}')
    print(f'  Confidence : {result["confidence"]}%')
    print(f'  Polarity   : {result["polarity"]:+.4f}')
    print(f'  Subjectivity: {result["subjectivity"]:.4f}')
    print(f'  Cleaned    : {result["clean_text"]}')
    print(f'{"="*50}')
else:
    print('No text entered.')